In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from PIL import Image
import matplotlib.patches as patches
from typing import Optional, List
import seaborn as sns
from scipy import signal
from matplotlib.widgets import RectangleSelector
from matplotlib.patches import Rectangle
import matplotlib.cm as cm
import json
import zipfile

import random
import gc

In [3]:
folder = '/content/drive/My Drive/Colab Notebooks/Infineon/'
rx_suffixes = ['_rx0.npy', '_rx2.npy']

# Get all files in the folder
all_files = os.listdir(folder)

# Group files by sample prefix (everything before '_rx0.npy')
samples = {}
for f in all_files:
    if f.endswith('_rx0.npy'):
        prefix = f.replace('_rx0.npy', '')
        samples[prefix] = [os.path.join(folder, prefix + s) for s in rx_suffixes]

print(f"Found {len(samples)} samples.")


Found 88 samples.


**PRE_PROCESSING**

In [4]:
def radar_pipeline(data, apply_window=True):
    """
    Full radar processing pipeline for data of shape (frames, chirps, range_bins, antennas)

    Parameters:
    - data: ndarray of shape (frames, chirps, range_bins, antennas)
    - apply_window: bool, apply Blackman-Harris window or not

    Returns:
    - processed: ndarray of shape (frames, range_bins, antennas) after chirp averaging
    """
    frames, chirps, range_bins, antennas = data.shape

    decluttered = np.zeros_like(data, dtype=np.complex64)

    # 1. Moving average over slow-time
    for f in range(frames):
        start = max(0, f - 10 + 1)
        clutter_estimate = np.mean(data[start:f+1, :, :, :], axis=0)
        decluttered[f, :, :, :] = data[f, :, :, :] - clutter_estimate

    # 2. Normalize ADC to [-1, 1]
    data_norm = 2 * decluttered / 4095 - 1.0

    # 3. DC removal per chirp, per antenna
    data_dc = data_norm - np.mean(data_norm, axis=2, keepdims=True)

    # 4. Apply range window (along range_bins axis)
    if apply_window:
        range_window = signal.windows.blackmanharris(range_bins).reshape(1, 1, range_bins, 1)
        data_windowed = data_dc * range_window
    else:
        data_windowed = data_dc

    # 5. Zero-padding along range_bins axis (double length)
    data_padded = np.pad(data_windowed, ((0,0),(0,0),(0,range_bins),(0,0)), 'constant')

    # 6. Range FFT along range_bins axis
    N_fft = data_padded.shape[2]
    range_fft = np.fft.fft(data_padded, axis=2) / N_fft
    # Keep only positive frequencies
    range_fft = 2 * range_fft[:, :, :range_bins, :]

    # 7. Average across chirps (axis=1), keep antennas separate
    range_profile_complex = np.mean(range_fft, axis=1)  # shape: (frames, range_bins, antennas)

    return range_profile_complex


**PRE-PROCESS SAMPLES**

In [5]:
# ---- Process all samples ----
all_processed_data = {}  # store all samples here
for sample_name, file_list in samples.items():
    # Load data for 3 antennas
    data_list = [np.load(f, allow_pickle=True) for f in file_list]
    combined_data = np.stack(data_list, axis=-1)  # shape: (frames, chirps, range_bins, antennas)

    # Apply radar pipeline
    processed = radar_pipeline(combined_data)

    # Store processed output
    all_processed_data[sample_name] = processed
    print(f"{sample_name}: processed shape {processed.shape}")


4_Still position_20250618-111410_Infineon: processed shape (1220, 128, 2)
8_Still position_20250618-110927_Infineon: processed shape (1220, 128, 2)
7_Still position_20250618-110649_Infineon: processed shape (1220, 128, 2)
8_Still position_20250618-112157_Infineon: processed shape (1220, 128, 2)
7_Still position_(250,45)_20250618-105902_Infineon: processed shape (1220, 128, 2)
2_Still position_20250618-110517_Infineon: processed shape (1220, 128, 2)
9_Still position_20250618-112319_Infineon: processed shape (1220, 128, 2)
14_Still position_20250618-111131_Infineon: processed shape (1220, 128, 2)
2-7-13_Still position_20250618-134845_Infineon: processed shape (1220, 128, 2)
13_Still position_20250618-112555_Infineon: processed shape (1220, 128, 2)
7_Still position_20250618-112028_Infineon: processed shape (1220, 128, 2)
3_Still position_20250618-111255_Infineon: processed shape (1220, 128, 2)
13_Still position_20250618-114029_Infineon: processed shape (1220, 128, 2)
7_Still position_2025

**Dataset Definition**

In [7]:
def extract_scenario_numbers(filename: str) -> List[str]:
    """
    Extracts a list of scenario numbers from a SR250 radar filename.

    Args:
        filename (str): The SR250 radar data filename (not the full path).

    Returns:
        List[str]: A list of strings, where each string is an extracted number.
                   Returns an empty list if no such pattern is found.
    """

    pattern = r'([0-9-]+)\s*_(?:Still|Moving) position_'

    match = re.search(pattern, filename)

    if match:
        numbers_str = match.group(1) # Get the captured sequence
        # Split the captured string by '-' and then use a list comprehension
        # to filter out any empty strings that result from multiple or trailing hyphens.
        return [num for num in numbers_str.split('-') if num]
    else:
        return []

# Define the map (position) -> (x_min, x_max, y_min, y_max) bigger bouding boxes for 96 by 96 image

In [17]:
# Define the map (position) -> (x_min, x_max, y_min, y_max)
position_map = {
    "2": (18.5, 26.5, 34, 45),
    "3": (-4, 4, 34, 45),
    "4": (-26.5, -18.5, 34, 45),
    "7": (18.5, 26.5, 59.5, 70.5),
    "8": (-4, 4, 59.5, 70.5),
    "9": (-26.5, -18.5, 59.5, 70.5),
    "12": (18.5, 26.5, 85, 96),
    "13": (-4, 4, 85, 96),
    "14": (-26.5, -18.5, 85, 96)
}

In [18]:
# Default label structure
DEFAULT_LABELS_STRUCTURE = {
    "version": 1,
    "type": "bounding-box-labels",
    "boundingBoxes": {}
}

**Beamforming and angles**

In [19]:
!rm -r ei_dataset_avg

In [20]:
window_size = 20   # number of frames to average
step_size = 20     # slide window step
output_dir = "ei_dataset_avg"
os.makedirs(output_dir, exist_ok=True)

In [21]:
#Beamforming weights
def steering_vector(N_rx, d, wavelength, angles_deg):
    """
    Compute steering vectors for a uniform linear array (ULA)

    Parameters:
    - N_rx: number of antennas
    - d: antenna spacing (meters)
    - wavelength: signal wavelength (meters)
    - angles_deg: 1D array of angles to scan (degrees)

    Returns:
    - steering_mat: shape (len(angles_deg), N_rx), complex steering vectors
    """
    angles_rad = np.deg2rad(angles_deg)
    n = np.arange(N_rx)  # antenna indices
    # Steering vector for each angle: exp(-j*2*pi*d/lambda * n * sin(theta))
    steering_mat = np.exp(-1j * 2 * np.pi / wavelength * d * np.outer(np.sin(angles_rad), n))
    return steering_mat

In [22]:
#Compute beamforming
def perform_beamforming(cleaned_data,steering_matrix):
    """
    cleaned_data: shape (N_frames, N_bins, N_antennas)
    steering_matrix: shape (N_angles, N_antennas)
    """
    N_frames, N_bins, _ = cleaned_data.shape
    N_angles = steering_matrix.shape[0]
   # Initialize output
    beamformed_cube = np.zeros((N_frames, N_bins,N_angles), dtype=np.complex128)
    # Perform beamforming across all frames and bins
    for t in range(N_frames):
        for r in range(N_bins):
            # For time frame t and range bin r extract the vector signals across all antennas(in this case 3)
            x = cleaned_data[t, r,: ]
            # Compute beamformed signals across all angles
            beamformed_cube[t, r, :] = np.conj(steering_matrix) @ x / cleaned_data.shape[2] # shape (N_angles,)
    return beamformed_cube

**Apply the Pipeline**

In [23]:
consolidated_labels_data = DEFAULT_LABELS_STRUCTURE.copy()
all_bboxes_for_json = {}

In [24]:
# Parameters
c = 299792458  # speed of light in m/s
f = 60e9  # center frequency of TSRR250 (Hz)
wavelength = c / f
d = 2.5e-3  # half wavelength antenna spacing

# Angle scan from -45 to 45 degrees, 181 points
angles = np.linspace(-45, 45, 90)

N_rx=2 # number of antennas

# Get steering vectors and perform beamforming
steering_mat = steering_vector(N_rx, d, wavelength, angles)
for sample_name, processed in all_processed_data.items():
   # 1. Extract labels from the filename (these labels are static for all windows of this file)
   labels = extract_scenario_numbers(sample_name)

   #2. Apply beamforming
   bf_data = perform_beamforming(processed,steering_mat)
   if bf_data is None:
    print(f"Skipping processing of {sample_name} due to beamforming failure.")
    continue

   num_total_frames = bf_data.shape[0]
   print(f"Total time frames available for {sample_name}: {num_total_frames}")

   # 4. Iterate through sliding windows and generate images and JSON entries
   # The loop iterates from 0 up to the last possible start frame that allows a full WINDOW_SIZE
   num_frames, num_bins, num_angles = bf_data.shape

   # Averaged windows
   for start in range(0, num_frames - window_size + 1, step_size):
      end = start + window_size
      window_avg = np.mean(np.abs(bf_data[start:end, :, :]), axis=0)

      # --- Reverse the range axis so that near range is at the bottom ---
      window_avg_flipped = window_avg[::-1, :]

      # --- Convert to dB ---
      magnitude_db = 20 * np.log10(window_avg_flipped + 1e-6)

      # --- Normalize for colormap ---
      normed = (magnitude_db - magnitude_db.min()) / (magnitude_db.max() - magnitude_db.min() + 1e-6)
      cmap = plt.colormaps["inferno"]
      img = (cmap(normed)[:, :, :3] * 255).astype(np.uint8)

      # --- Resize to 96x96 ---
      im_96 = Image.fromarray(img).resize((96, 96), Image.BICUBIC)

      # 3. Create the common bounding box list for this original file
      rows, cols = window_avg_flipped.shape
      bounding_boxes = []
      for label_str in labels:
        if not label_str:
          continue
        if label_str in position_map:
          x_min, x_max, y_min, y_max = position_map[label_str]

          scale_x = 96 / cols
          scale_y = 96 / rows
          box_left = int((x_min - angles[0]) * scale_x)
          box_top = int((rows - y_max) * scale_y)  # because image was flipped
          box_width = int((x_max - x_min) * scale_x)
          box_height = int((y_max - y_min) * scale_y)

          bbox ={
              "label": "human",
              "x": box_left,
              "y": box_top,
              "width": box_width,
              "height": box_height
          }
          bounding_boxes.append(bbox)
        else:
            print(f"Warning: Label '{label_str}' extracted from '{sample_name}' "
                  f"not found in positions_map. Skipping bounding box for it.")

      # --- Save image ---
      file_name = f"{sample_name}_{start}_{end}_64x64.png"
      file_path = os.path.join(output_dir, file_name)
      im_96.save(file_path)

      all_bboxes_for_json[file_name] = bounding_boxes

   # 5. Populate the consolidated JSON structure and write it to a single file
   consolidated_labels_data["boundingBoxes"] = all_bboxes_for_json

   final_labels_filepath = os.path.join(output_dir, "bounding_boxes.labels")
   try:
        with open(final_labels_filepath, 'w') as f:
            json.dump(consolidated_labels_data, f, indent=2)
        print(f"\n--- Successfully wrote all consolidated bounding box labels to: {final_labels_filepath} ---")
   except Exception as e:
        print(f"\n--- Error writing consolidated JSON file {final_labels_filepath}: {e} ---")




Total time frames available for 4_Still position_20250618-111410_Infineon: 1220

--- Successfully wrote all consolidated bounding box labels to: ei_dataset_avg/bounding_boxes.labels ---
Total time frames available for 8_Still position_20250618-110927_Infineon: 1220

--- Successfully wrote all consolidated bounding box labels to: ei_dataset_avg/bounding_boxes.labels ---
Total time frames available for 7_Still position_20250618-110649_Infineon: 1220

--- Successfully wrote all consolidated bounding box labels to: ei_dataset_avg/bounding_boxes.labels ---
Total time frames available for 8_Still position_20250618-112157_Infineon: 1220

--- Successfully wrote all consolidated bounding box labels to: ei_dataset_avg/bounding_boxes.labels ---
Total time frames available for 7_Still position_(250,45)_20250618-105902_Infineon: 1220

--- Successfully wrote all consolidated bounding box labels to: ei_dataset_avg/bounding_boxes.labels ---
Total time frames available for 2_Still position_20250618-110

In [ ]:
!ls ei_dataset_avg

In [26]:
from google.colab import files
# Folder where your images and JSON are stored
output_dir = "ei_dataset_avg"  # change this to your folder in Colab

# Name of the zip file
zip_filename = os.path.join(output_dir, "all_images_and_labels.zip")

# Collect all .png and .json files
files_to_zip = [f for f in os.listdir(output_dir) if f.endswith(".png") or f.endswith(".labels")]

# Create the zip
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for f in files_to_zip:
        zipf.write(os.path.join(output_dir, f), arcname=f)

print(f"Created zip file: {zip_filename}")

# Download the zip to your PC
files.download(zip_filename)

Created zip file: ei_dataset_avg/all_images_and_labels.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>